In [1]:
import pandas as pd
import numpy as np
from scipy.io import loadmat
from pathlib import Path
import mne

In [2]:
data_folder = Path('../Data/')
raw_data_folder = Path('../Data/raw_data/')

In [3]:
data_labels = pd.read_csv(data_folder / "preprocessed_data.csv")

In [4]:
data_labels.head()

,subject,trial,kind,file_name,stress_level
0,1,1,Relax,Relax_sub_1_trial1.mat,0
1,1,1,Mirror,Mirror_image_sub_1_trial1.mat,3
2,1,1,Arithmetic,Arithmetic_sub_1_trial1.mat,6
3,1,1,Stroop,Stroop_sub_1_trial1.mat,3
4,1,2,Mirror,Mirror_image_sub_1_trial2.mat,5


In [5]:
data_labels.shape

(480, 5)

In [6]:
data_labels["kind"].value_counts()

kind
Relax         120
Mirror        120
Arithmetic    120
Stroop        120
Name: count, dtype: int64

In [7]:
stress_df = data_labels[data_labels["kind"] != "Relax"].copy()

In [8]:
stress_df["kind"].value_counts()

kind
Mirror        120
Arithmetic    120
Stroop        120
Name: count, dtype: int64

## process each file

In [9]:
SFREQ = 128
N_CHANNELS = 32

LOW_FREQ = 0.5
HIGH_FREQ = 45

In [50]:
def process_file(filepath: str):
    mat = loadmat(filepath)
    data = mat['Data'] # data.shape => (32, 3200) -> 32 channels, 3200 samples

    ch_names = [
        f"EEG{i:03d}"
        for i in range(1, N_CHANNELS + 1)
    ]


    info = mne.create_info(
        ch_names=ch_names,
        sfreq=SFREQ,
        ch_types="eeg"
    )

    raw = mne.io.RawArray(
        data,
        info,
        verbose=False
    )

    raw_filtered = raw.copy()

    raw_filtered.filter(
        l_freq=LOW_FREQ,
        h_freq=HIGH_FREQ,
        verbose=False
    )


    ica = mne.preprocessing.ICA(
        n_components=15,
        random_state=42,
        max_iter=1000,
        method="picard",
        fit_params={"tol": 0.01}
    )

    ica.fit(
        raw_filtered,
        verbose=False,
    )


    raw_clean = raw_filtered.copy()


    ica.apply(
        raw_clean,
        verbose=False
    )

    epochs = mne.make_fixed_length_epochs(
        raw_clean,
        duration=2.0,
        overlap=1.0,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()
    file_path = str(filepath).split("/")[-1]


    n_epochs = X.shape[0]

    metadata = data_labels[data_labels['file_name'] == file_path][['subject', 'trial', 'kind', 'stress_level']]
    metadata = pd.concat([metadata] * n_epochs, ignore_index=True)

    return X, metadata

In [52]:
filepath = raw_data_folder / "Mirror_image_sub_1_trial1.mat"
X, metadata = process_file(filepath)
metadata

,subject,trial,kind,stress_level
0,1,1,Mirror,3
1,1,1,Mirror,3
2,1,1,Mirror,3
3,1,1,Mirror,3
4,1,1,Mirror,3
5,1,1,Mirror,3
6,1,1,Mirror,3
7,1,1,Mirror,3
8,1,1,Mirror,3
9,1,1,Mirror,3


In [53]:
print(X.shape)

(24, 32, 256)


## Build The Entire Dataset

In [57]:
all_X = []
all_metadata = []

for _, row in data_labels.iterrows():
    file_path = raw_data_folder / row['file_name']
    X, metadata = process_file(file_path)

    all_X.append(X)
    all_metadata.append(metadata)


# Combine all EEG epochs
X_full = np.concatenate(
    all_X,
    axis=0
)

# Combine metadata
metadata_full = pd.concat(
    all_metadata,
    ignore_index=True
)


In [59]:
print("X_full:", X_full.shape)
print("metadata_full:", metadata_full.shape)

print(metadata_full.head())

X_full: (11520, 32, 256)
metadata_full: (11520, 4)
   subject  trial   kind  stress_level
0        1      1  Relax             0
1        1      1  Relax             0
2        1      1  Relax             0
3        1      1  Relax             0
4        1      1  Relax             0


In [68]:
metadata_full["stress_level"].value_counts().sort_index()

stress_level
0     2880
1      456
2      600
3     1392
4     1296
5     1848
6     1368
7      768
8      576
9      240
10      96
Name: count, dtype: int64

In [69]:
print(
    metadata_full.groupby("kind")["stress_level"].agg(
        ["count", "mean", "min", "max"]
    )
)

            count      mean  min  max
kind                                 
Arithmetic   2880  5.441667    1   10
Mirror       2880  4.941667    1    9
Relax        2880  0.000000    0    0
Stroop       2880  4.050000    1    9


## Save

In [70]:
np.save(
    "../Data/processed/X.npy",
    X_full
)

metadata_full.to_csv(
    "../Data/processed/metadata.csv",
    index=False
)